In [6]:
import csv
import glob
import os
from pprint import pprint

In [ ]:
def load_convergence_data(base_dir):
    """load rows grouped by folder."""
    rows_by_folder = {}
    for folder in [f"s{i}" for i in range(10)]:
        rows = []
        for file in glob.glob(os.path.join(base_dir, folder, "version_vector_states.csv")):
            with open(file, newline="") as f:
                reader = csv.reader(f)
                try:
                    _ = next(reader)
                except StopIteration:
                    continue
                for reader_row in reader:
                    ts = int(reader_row[0])
                    values = list(map(int, reader_row[1:]))
                    rows.append((ts, values))
        rows_by_folder[folder] = rows
    return rows_by_folder

def first_occurrence_thresholds(rows, k):
    """return list of (threshold, timestamp) for first occurrence of each value in column k."""
    seen = set()
    result = []
    for ts, values in rows:
        val = values[k]
        if val == 0:
            continue
        if val not in seen:
            seen.add(val)
            result.append((val, ts))
    return result


def process_folder_first_occurrence(data, k):
    """process folder s{k} using first occurrence per threshold in column k."""
    results = []

    thresholds = first_occurrence_thresholds(data[f"s{k}"], k)

    for val, base_ts in thresholds:
        max_ts = 0
        for folder, rows in data.items():
            if folder == f"s{k}":
                continue
            candidate_ts = [ts2 for ts2, vals2 in rows if vals2[k] >= val]
            min_ts = min(candidate_ts) if candidate_ts else None
            if min_ts == None: # This did not converge, so we exclude this
                max_ts = None
                break
            if min_ts > max_ts:
                max_ts = min_ts
        if max_ts != None:
            results.append({
                "from_folder": f"s{k}",
                "base_timestamp": base_ts,
                "threshold": val,
                "max_timestamp_other_folders": max_ts,
                "time_difference(us)": max_ts - base_ts
            })

    return results

def get_convergence_times_dict(base_dir):
    data = load_convergence_data(base_dir)
    results = {}

    for i in range(0, 10):
        results[i] = process_folder_first_occurrence(data, i)
            
    return results


In [ ]:

for run in range(1, 6):
    res = get_convergence_times_dict(f"../benchmarks/exp_network_ring_200_pull_one_run{run}")
pprint(res)

{0: [{'base_timestamp': 1759245122166741,
      'from_folder': 's0',
      'max_timestamp_other_folders': 1759245122649506,
      'threshold': 1,
      'time_difference(us)': 482765},
     {'base_timestamp': 1759245122398659,
      'from_folder': 's0',
      'max_timestamp_other_folders': 1759245122938312,
      'threshold': 2,
      'time_difference(us)': 539653},
     {'base_timestamp': 1759245122682996,
      'from_folder': 's0',
      'max_timestamp_other_folders': 1759245123534235,
      'threshold': 3,
      'time_difference(us)': 851239},
     {'base_timestamp': 1759245122933726,
      'from_folder': 's0',
      'max_timestamp_other_folders': 1759245123762226,
      'threshold': 4,
      'time_difference(us)': 828500},
     {'base_timestamp': 1759245123325389,
      'from_folder': 's0',
      'max_timestamp_other_folders': 1759245123822301,
      'threshold': 5,
      'time_difference(us)': 496912},
     {'base_timestamp': 1759245123585746,
      'from_folder': 's0',
      'max_